# MILP GNN Baseline Verification

Quick sanity checks for feature dimensions, data collection, model forward pass, and CSR conversion.

In [1]:
import sys
sys.path.insert(0, '../src')

import os
import pickle
import numpy as np
import torch

from data_collection import create_instance_generator, create_branching_environment, collect_samples, verify_samples
from gnn_model import BipartiteGNN
from export import verify_csr_format


In [2]:
# 1) Feature dimensions
gen = create_instance_generator(n_rows=500, n_cols=1000, density=0.05, seed=0)
env = create_branching_environment(use_strong_branching=False)
obs, action_set, _, done, _ = env.reset(next(gen))

print('var features:', obs.variable_features.shape)
print('con features:', obs.row_features.shape)
print('edge idx:', obs.edge_features.indices.shape)

assert obs.variable_features.shape[1] == 19
assert obs.row_features.shape[1] == 5


pressed CTRL-C 1 times (5 times for forcing termination)
var features: (237, 19)
con features: (539, 5)
edge idx: (2, 13022)


In [ ]:
# 2) Data collection (small)
samples = collect_samples(num_samples=100, max_steps_per_instance=25, verbose=True)
stats = verify_samples(samples)
stats


Feature dimensions: var=19, con=5
pressed CTRL-C 1 times (5 times for forcing termination)


In [ ]:
# 3) Model forward pass
s = samples[0]
model = BipartiteGNN(var_dim=s['variable_features'].shape[1], con_dim=s['constraint_features'].shape[1], emb_dim=64)
model.eval()

with torch.no_grad():
    scores = model(
        torch.tensor(s['variable_features']),
        torch.tensor(s['constraint_features']),
        torch.tensor(s['edge_indices']),
        torch.tensor(s['edge_values']),
        torch.tensor(s['action_mask']),
    )

print(scores.shape, 'valid:', int(s['action_mask'].sum()))


In [ ]:
# 4) CSR conversion check
ok = verify_csr_format(s['edge_indices'], s['edge_values'], s['constraint_features'].shape[0])
assert ok
